# Backtesting and Trading Signal Generation

This notebook implements:
1. Trading signal generation based on model predictions
2. Backtesting framework for strategy evaluation
3. Performance comparison between different models

We'll evaluate the trading performance of our baseline models and compare their returns.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
sys.path.append('../notebooks')
from notebook_00_data_preparation import create_stock_features
from notebook_01_baseline_models import evaluate_model

## Trading Signal Generation

In [ ]:
def generate_signals(predictions, actual_prices, threshold=0.001):
    """Generate trading signals based on predicted price movements
    
    Args:
        predictions (np.array): Model predictions
        actual_prices (np.array): Actual prices
        threshold (float): Minimum price movement threshold for signal generation
        
    Returns:
        pd.DataFrame: DataFrame with trading signals
    """
    signals = pd.DataFrame({
        'actual_price': actual_prices,
        'predicted_price': predictions
    })
    
    # Calculate predicted returns
    signals['predicted_return'] = (signals['predicted_price'] - signals['actual_price']) / signals['actual_price']
    
    # Generate signals
    signals['position'] = 0  # 0: Hold, 1: Buy, -1: Sell
    signals.loc[signals['predicted_return'] > threshold, 'position'] = 1
    signals.loc[signals['predicted_return'] < -threshold, 'position'] = -1
    
    return signals

## Backtesting Framework

In [ ]:
class BacktestStrategy:
    """Backtesting framework for evaluating trading strategies"""
    def __init__(self, initial_capital=100000, transaction_cost=0.001):
        self.initial_capital = initial_capital
        self.transaction_cost = transaction_cost
        self.reset()
        
    def reset(self):
        """Reset the backtest state"""
        self.capital = self.initial_capital
        self.position = 0
        self.trades = []
        self.equity_curve = [self.initial_capital]
        
    def execute_trade(self, price, signal, timestamp):
        """Execute a trade based on the signal"""
        if signal != self.position:
            # Close existing position
            if self.position != 0:
                cost = abs(self.position) * price * self.transaction_cost
                self.capital += self.position * price - cost
                self.trades.append({
                    'timestamp': timestamp,
                    'type': 'close',
                    'price': price,
                    'pnl': self.position * price - cost
                })
            
            # Open new position
            if signal != 0:
                cost = abs(signal) * price * self.transaction_cost
                self.capital -= signal * price + cost
                self.trades.append({
                    'timestamp': timestamp,
                    'type': 'open',
                    'price': price,
                    'position': signal
                })
            
            self.position = signal
            self.equity_curve.append(self.capital)
            
    def run_backtest(self, signals):
        """Run backtest on trading signals
        
        Args:
            signals (pd.DataFrame): DataFrame with trading signals
            
        Returns:
            dict: Backtest results and performance metrics
        """
        self.reset()
        
        for timestamp, row in signals.iterrows():
            self.execute_trade(row['actual_price'], row['position'], timestamp)
            
        # Calculate performance metrics
        equity_curve = pd.Series(self.equity_curve)
        returns = equity_curve.pct_change().dropna()
        
        results = {
            'total_return': (self.capital - self.initial_capital) / self.initial_capital,
            'sharpe_ratio': np.sqrt(252) * returns.mean() / returns.std(),
            'max_drawdown': (equity_curve / equity_curve.cummax() - 1).min(),
            'num_trades': len(self.trades),
            'equity_curve': equity_curve
        }
        
        return results

## Performance Analysis

In [ ]:
def analyze_performance(backtest_results, model_name):
    """Analyze and visualize trading strategy performance
    
    Args:
        backtest_results (dict): Results from backtesting
        model_name (str): Name of the model being analyzed
    """
    print(f"\nPerformance Analysis for {model_name}:")
    print(f"Total Return: {backtest_results['total_return']:.2%}")
    print(f"Sharpe Ratio: {backtest_results['sharpe_ratio']:.2f}")
    print(f"Maximum Drawdown: {backtest_results['max_drawdown']:.2%}")
    print(f"Number of Trades: {backtest_results['num_trades']}")
    
    # Plot equity curve
    plt.figure(figsize=(12, 6))
    plt.plot(backtest_results['equity_curve'])
    plt.title(f'Equity Curve - {model_name}')
    plt.xlabel('Trade Number')
    plt.ylabel('Portfolio Value')
    plt.grid(True)
    plt.show()

## Example Usage

In [ ]:
# Example: Run backtesting on model predictions
if __name__ == "__main__":
    # Load model predictions
    # predictions = ... (load predictions from baseline models)
    
    # Generate trading signals
    # signals = generate_signals(predictions, actual_prices)
    
    # Run backtest
    # backtest = BacktestStrategy()
    # results = backtest.run_backtest(signals)
    
    # Analyze performance
    # analyze_performance(results, 'Model Name')